# 02 — Calibration and Policy Layer
Improved path (contrast with notebook 02):
1. 1D-CNN with **value + mask + dt** channels (honors missing data)
2. Temperature scaling on a held-out calibration set
3. **PolicyLayer**: expected cost + dwell + hysteresis + abstain

In [ ]:
import numpy as np

from sentinelai.baseline.weak_controls import forward_fill_only, summarize_weak, weak_decide
from sentinelai.data.skab import load_all_scenarios
from sentinelai.data.windows import inject_gaps, make_windows, temporal_split
from sentinelai.models.cnn_detector import predict_logits, predict_proba, train_detector
from sentinelai.policy.decision import PolicyLayer, CostMatrix
from sentinelai.uncertainty.calibration import (
    calibrate_probs,
    expected_calibration_error,
    fit_temperature,
    reliability_curve
)
from sentinelai.uncertainty.conformal import ConformalPredictor

# TO DO: better handling of gaps - transformer model?

In [ ]:
df = load_all_scenarios(("valve1", "valve2"))

# gapped = inject_gaps(df)
# batch = make_windows(gapped)

filled = forward_fill_only(df)
batch = make_windows(filled)
train, cal, test = temporal_split(batch)


print(len(train.labels), len(cal.labels), len(test.labels))

In [ ]:
df = load_all_scenarios(("valve1", "valve2"))

filled = forward_fill_only(df)
batch = make_windows(filled)
train, cal, test = temporal_split(batch)
print(len(train.labels), len(cal.labels), len(test.labels))



In [ ]:
result = train_detector(train, val=cal, epochs=100, values_only=True)
model = result.model
print(f"Final train loss: {result.train_losses[-1]:.4f}")
print(f"Final val loss: {result.val_losses[-1]:.4f}")


In [ ]:
import matplotlib.pyplot as plt

plt.plot(result.train_losses, marker='o', label='Train')
plt.plot(result.val_losses, marker='o', label='Validation')
plt.title("Train/Validation Loss During Training")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
p_raw = predict_proba(model, test, values_only=True)

# F1 score
import sklearn.metrics  



f1_raw_score = sklearn.metrics.f1_score(test.labels, p_raw > 0.5)
print(f"F1 raw score: {f1_raw_score:.3f}")




In [ ]:
logits_cal = predict_logits(model, cal, values_only=True)
print(f"logits_cal.shape: {logits_cal.shape}")
print(f"cal.labels.shape: {cal.labels.shape}")

T = fit_temperature(logits_cal, cal.labels)
print(f"Temperature T = {T:.3f}")

p_cal = calibrate_probs(predict_logits(model, test, values_only=True), T)

print(f"ECE raw: {expected_calibration_error(p_raw, test.labels):.3f}")
print(f"ECE calibrated: {expected_calibration_error(p_cal, test.labels):.3f}")

In [ ]:

centers, confs_raw, accs_raw = reliability_curve(p_raw, test.labels)
centers, confs_cal, accs_cal = reliability_curve(p_cal, test.labels)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], 'k--', label='perfect calibration')
ax.plot(confs_raw, accs_raw, 'o-', label='Raw (uncalibrated)')
ax.plot(confs_cal, accs_cal, 'o-', label='Calibrated')
ax.set_xlabel('Mean confidence ')
ax.set_ylabel('Accuracy')
ax.set_title('Reliability diagram — uncalibrated CNN scores')
ax.legend()
plt.show()

In [ ]:
policy1 = PolicyLayer(cost=CostMatrix(cost_fn=100.0, cost_fp=1))
policy1.reset()
records = []
for i in range(len(test.labels)):
    rec = policy1.decide(
        p_hazard_raw=float(p_raw[i]),
        p_hazard_calibrated=float(p_cal[i]),
        coverage=float(test.coverage[i]),
    )
    records.append(rec)

actions = [r.action for r in records]
print({a: actions.count(a) for a in set(actions)})

# Check accuracy of policy
from sklearn.metrics import accuracy_score

# Extract true labels and predicted actions
true_labels = test.labels
is_alarm1 = [r.action == "alarm" for r in records]

# Calculate accuracy
policy_accuracy1 = accuracy_score(true_labels, is_alarm1)
print(f"Policy1 accuracy: {policy_accuracy1:.3f}")




In [ ]:

# Reduce cost of missing alarm
policy2 = PolicyLayer(cost=CostMatrix(cost_fn=1.0, cost_fp=1))
policy2.reset()
records = []
for i in range(len(test.labels)):
    rec = policy2.decide(
        p_hazard_raw=float(p_raw[i]),
        p_hazard_calibrated=float(p_cal[i]),
        coverage=float(test.coverage[i]),
    )
    records.append(rec)

actions = [r.action for r in records]
print({a: actions.count(a) for a in set(actions)})

# Check accuracy of policy
from sklearn.metrics import accuracy_score

# Extract true labels and predicted actions
true_labels = test.labels
is_alarm2 = [r.action == "alarm" for r in records]

# Calculate accuracy
policy_accuracy2 = accuracy_score(true_labels, is_alarm2)
print(f"Policy2 accuracy: {policy_accuracy2:.3f}")

